# emg3d hacks: Compute gradient of subsections, and clean gradient-related data

See https://github.com/emsig/emg3d/issues/357

In [1]:
import emg3d
import numpy as np

## A survey/model/simulation

Adusted from the example [Adjoint-state vs. FD gradient](https://emsig.xyz/emg3d-gallery/gallery/comparisons/as_vs_fd_gradient.html)

In [2]:
survey = emg3d.surveys.Survey(
    name='Gradient Test-Survey',
    sources=emg3d.TxElectricDipole((-1600, 0, -1950, 0, 0)),
    receivers=emg3d.RxElectricPoint((1600, 0, -2000, 0, 0)),
    frequencies=[0.5, 1.0, 2.0],
    noise_floor=1e-15,
    relative_error=0.05,
)

# Create a simple block model.
hx = np.array([1000, 1500, 1000, 1500, 1000])
hy = np.array([1000, 1500, 1000, 1500, 1000])
hz = np.array([1800., 200., 200., 200., 300., 300., 2000.,  500.])
model_grid = emg3d.TensorMesh(
        [hx, hy, hz], origin=np.array([-3000, -3000, -5000]))

# Initiate model with conductivities of 1 S/m.
model = emg3d.Model(
        model_grid, np.ones(model_grid.n_cells), mapping='Conductivity')
model.property_x[:, :, -1] = 1e-8  # Add air layer.
model.property_x[:, :, -2] = 3.33  # Add seawater layer.
model_bg = model.copy()  # Make a copy for the background model.

# Add three blocks.
model.property_x[1, 1:3, 1:3] = 0.02
model.property_x[3, 2:4, 2:4] = 0.01
model.property_x[2, 1:4, 4] = 0.005

# Gridding options.
gridding_opts = {
    'frequency': survey.frequencies['f-1'],
    'properties': [3.33, 1, 1, 3.33],
    'center': (0, 0, -2000),
    'min_width_limits': 100,
    'domain': ([-2000, 2000], [-2000, 2000], [-3200, -2000]),
    'mapping': model.map,
    'center_on_edge': True,
}

data_grid = emg3d.construct_mesh(**gridding_opts)

## Generate 'observed' data

In [3]:
# Define a simulation for the data.
simulation_data = emg3d.simulations.Simulation(
    name='Data for Gradient Test',
    survey=survey,
    model=model.interpolate_to_grid(data_grid),
    gridding='same',  # Same grid as for input model.
    max_workers=4,
)

# Simulate the data and store them as observed.
simulation_data.compute(observed=True)

Compute efields            0/3  [00:00]

## Generate gradient

In [4]:
# Computational grid (min_width 200 instead of 100).
comp_grid_opts = {**gridding_opts, 'min_width_limits': 200}
comp_grid = emg3d.construct_mesh(**comp_grid_opts)

# Interpolate the background model onto the computational grid.
comp_model = model_bg.interpolate_to_grid(comp_grid)

# AS gradient simulation.
sim = emg3d.simulations.Simulation(
    name='AS Gradient Test',
    survey=survey,
    model=comp_model,
    gridding='same',  # Same grid as for input model.
    max_workers=4,    # For parallel workers, adjust if you have more.
    receiver_interpolation='linear',  # For proper adjoint-state gradient
)

gsim = sim.gradient

Compute efields            0/3  [00:00]

Back-propagate            0/3  [00:00]

## 1. Selections

This functionality should really go into a function `Simulation.select()` or similar (not in the hackish way below, but properly).

With [survey.select](https://emg3d.emsig.xyz/en/stable/api/emg3d.surveys.Survey.html#emg3d.surveys.Survey.select), you can choose any combination of source(s), frequency(ies), and receiver(s).

This is a hack. We keep the `Simulation` with all fields, but replace the survey with a subset to extract the gradient. Use with caution, but within this scope, it works.

In [5]:
# We have to re-set the previously computed gradient
sim._gradient = None
# Now we set the survey to a selection of it
sim.survey = survey.select(sources='TxED-1', frequencies='f-1')
# We re-compute the gradient. It will re-use the already computed e-fields and backwards-fields
gsim1 = sim.gradient

Back-propagate            0/1  [00:00]

In [6]:
sim._gradient = None
sim.survey = survey.select(frequencies=['f-2', 'f-3'])
gsim2 = sim.gradient

Back-propagate            0/2  [00:00]

## Assert the two subsets add up to the same as the complete gradient

In [7]:
np.allclose(gsim, gsim1 + gsim2, atol=0, rtol=1e-7)

True

# 2. Clean gradient-related stuff

Similarly as above, you can re-use already computed e-fields.

This functionality should really go into a function `Simulation.clean('gradient')` or similar (not in the hackish way below, but properly).

```python
newsim = sim.copy('all')                    # Copy entire simulation
newsim.survey = survey_with_other_obs_data  # Replace the data
newsim._gradient = None                     # Reset computed gradient
newsim._misfit = None                       # Reset computed misfit
if hasattr(newsim, '_dict_bfield'):         # Reset computed backprogagated fields
    delattr(newsim, '_dict_bfield')

newsim.gradient
```

In [8]:
emg3d.Report()

--------------------------------------------------------------------------------
  Date: Fri Mar 27 15:33:35 2026 UTC

                OS : Linux (Ubuntu 24.04)
            CPU(s) : 16
           Machine : x86_64
      Architecture : 64bit
               RAM : 93.6 GiB
       Environment : Jupyter
       File system : ext4

  Python 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:38:13) [GCC
  12.3.0]

             numpy : 2.3.5
             scipy : 1.16.0
             numba : 0.63.1
             emg3d : 1.8.8.dev3+gf35cb50eb
           empymod : 2.6.0
            xarray : 2024.11.0
        discretize : 0.12.0
              h5py : 3.13.0
        matplotlib : 3.9.1
              tqdm : 4.67.1
           IPython : 8.31.0
--------------------------------------------------------------------------------